In [1]:
import asyncio
import os
import pandas as pd
from tabulate import tabulate
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from langchain_core.tools import StructuredTool as _StructuredTool
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from typing import TypedDict
import json
import re
from difflib import get_close_matches

# -----------------------------
# Patch: StructuredTool shim to avoid Pydantic __fields__ deprecation
# -----------------------------
class StructuredTool(_StructuredTool):
    @classmethod
    def from_function(cls, func, **kwargs):
        tool = super().from_function(func, **kwargs)
        # Touch model_fields eagerly so downstream code won't access __fields__
        _ = getattr(tool.args_schema, "model_fields", {})
        return tool

# -----------------------------
# Define state schema (optional)
# -----------------------------
class AgentState(TypedDict):
    user_input: str
    agent_response: str

# -----------------------------
# Initialize LLM
# -----------------------------
model = init_chat_model(
    "ollama:qwen2.5:3b",
    model_kwargs={"temperature": 0, "num_predict": 512},  # a bit more room to reason
    base_url="http://ollama:11434"
)

# -----------------------------
# Sync wrapper for MCP tools
# -----------------------------
def make_sql_tool_wrapper(tool_name, client):
    def wrapper(sql: str = "") -> str:
        try:
            sql = (sql or "").rstrip(";").strip()
            loop = asyncio.get_event_loop()
            TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", 60))
            coro = client.call_tool(tool_name, {"sql": sql})
            result = loop.run_until_complete(asyncio.wait_for(coro, timeout=TIMEOUT))
            if hasattr(result, 'data') and result.data:
                return json.dumps(result.data)
            elif hasattr(result, "structuredContent") and result.structuredContent:
                return json.dumps(result.structuredContent)
            elif hasattr(result, "content") and result.content:
                return result.content[0].text if isinstance(result.content, list) else str(result.content)
            else:
                return str(result)
        except asyncio.TimeoutError:
            return f"❌ Timeout calling {tool_name}"
        except Exception as e:
            return f"❌ Error calling {tool_name}: {str(e)}"
    return wrapper

def make_noarg_tool_wrapper(tool_name, client):
    def wrapper() -> str:
        try:
            loop = asyncio.get_event_loop()
            TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", 60))
            coro = client.call_tool(tool_name, {})  # <-- no args
            result = loop.run_until_complete(asyncio.wait_for(coro, timeout=TIMEOUT))
            if hasattr(result, 'data') and result.data:
                return json.dumps(result.data)
            elif hasattr(result, "structuredContent") and result.structuredContent:
                return json.dumps(result.structuredContent)
            elif hasattr(result, "content") and result.content:
                return result.content[0].text if isinstance(result.content, list) else str(result.content)
            else:
                return str(result)
        except asyncio.TimeoutError:
            return f"❌ Timeout calling {tool_name}"
        except Exception as e:
            return f"❌ Error calling {tool_name}: {str(e)}"
    return wrapper

# -----------------------------
# Render helpers
# -----------------------------
def display_tables_list(data: dict):
    """Render list_tables response."""
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("ℹ️ No tables found.")
        return
    print("\n📚 Available tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    """Display table data in tabulated format (for query_db)."""
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        
        # Add "..." if rows or columns truncated
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(preview_cols):
            df["..."] = "..."
        
        print("\n📊 Query Result (preview):")
        print(tabulate(df, headers="keys", tablefmt="pretty"))
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
        
        # Show debug info if available
        debug_info = data.get("_debug_info", {})
        if debug_info:
            print(f"🔍 Debug: Query returned {debug_info.get('query_returned_rows', '?')} rows, "
                  f"True total: {debug_info.get('true_total_rows', '?')} rows")
    else:
        print("ℹ️ Query executed but returned no rows.")

# -----------------------------
# Utilities: parse tool messages
# -----------------------------
def _extract_json_from_text(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    # Look for top-level success object
    json_pattern = r'\{[^{}]*"success"[^{}]*\}'
    matches = re.findall(json_pattern, text)
    for match in matches:
        try:
            return json.loads(match)
        except Exception:
            continue
    # Heuristic balanced braces scan
    brace_count = 0
    start_idx = -1
    for i, ch in enumerate(text):
        if ch == '{':
            if brace_count == 0:
                start_idx = i
            brace_count += 1
        elif ch == '}':
            brace_count -= 1
            if brace_count == 0 and start_idx != -1:
                try:
                    parsed = json.loads(text[start_idx:i+1])
                    if isinstance(parsed, dict) and 'success' in parsed:
                        return parsed
                except Exception:
                    pass
    return None

def _collect_tool_payloads(result):
    """Return (list_tables_payload, query_db_payload, query_db_sql)."""
    messages = getattr(result, "messages", []) if hasattr(result, "messages") else result.get("messages", [])
    list_tables_payload = None
    query_db_payload = None
    query_db_sql = None

    for msg in messages:
        msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
        if 'tool' in msg_type:
            name = getattr(msg, 'name', None)
            content = getattr(msg, 'content', None)
            args = getattr(msg, 'args', {}) or getattr(msg, 'tool_input', {})
            if name == 'query_db' and args and 'sql' in args:
                query_db_sql = args['sql']
            if isinstance(content, str):
                payload = _extract_json_from_text(content)
            elif isinstance(content, list) and content:
                if isinstance(content[0], dict) and "text" in content[0]:
                    payload = _extract_json_from_text(content[0]["text"])
                else:
                    payload = _extract_json_from_text(str(content[0]))
            else:
                payload = None

            if payload and payload.get("success"):
                if name == 'list_tables':
                    list_tables_payload = payload
                elif name == 'query_db':
                    query_db_payload = payload
    return list_tables_payload, query_db_payload, query_db_sql

def _pick_closest_table(user_input: str, tables: list[str]) -> str | None:
    """Pick closest table from tables (prefers bare name; returns bare)."""
    tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", user_input.lower())
    if not tokens:
        return None
    candidate = tokens[-1]
    bare = [t.split(".", 1)[1] if "." in t else t for t in tables]
    best = get_close_matches(candidate, bare, n=1, cutoff=0.6)
    if best:
        return best[0]
    best = get_close_matches(candidate, tables, n=1, cutoff=0.6)
    if best:
        return best[0].split(".", 1)[1] if "." in best[0] else best[0]
    return None

# -----------------------------
# Define system prompt
# -----------------------------
system_prompt = """You are a SQL assistant that helps users query a PostgreSQL database.

WORKFLOW (you must follow this on every user query):
1) Call `list_tables` first (no arguments).
2) If the user's table name isn't an exact match, choose the closest table from the list (e.g., "department" → "departments").
3) Then call `query_db(sql: str)` using the exact table name. Prefer: SELECT * FROM <table> LIMIT 5;
4) If there is truly no close match, state that no similar table exists and show the available tables.

RULES:
- Only SELECT queries are allowed.
- Always show the SQL you executed.
- Never guess column or table names beyond what `list_tables` returns.
"""

# -----------------------------
# Main async function
# -----------------------------
async def main():
    transport = StreamableHttpTransport("http://mcp_server:8000/mcp")
    async with Client(transport=transport) as client:

        tools_list = await client.list_tools()
        tools = []

        for tool in tools_list:
            tool_name = getattr(tool, "name", None)
            description = getattr(tool, "description", f"MCP tool {tool_name}")

            if tool_name == "list_tables":
                tool_func = make_noarg_tool_wrapper(tool_name, client)
            else:
                tool_func = make_sql_tool_wrapper(tool_name, client)
            
            # Use our shimmed StructuredTool (no Pydantic deprecation warning)
            structured_tool = StructuredTool.from_function(
                func=tool_func,
                name=tool_name,
                description=description
            )
            tools.append(structured_tool)

        print("\n🔧 Available MCP Tools:")
        for t in tools:
            print(f"- {t.name}: {t.description}")

        agent_llm = create_react_agent(model=model, tools=tools)

        print("\n🤖 Agent ready! Ask SQL questions.")
        print("Type 'exit', 'quit', or 'q' to stop.")

        while True:
            try:
                user_input = input("\nEnter your SQL query: ").strip()
                if user_input.lower() in ("exit", "quit", "q"):
                    print("👋 Goodbye!")
                    break
                if not user_input:
                    continue

                print("🤔 Thinking...")

                # One-shot with fresh system+user only (no chat history)
                result = await agent_llm.ainvoke({"messages": [("system", system_prompt), ("user", user_input)]})

                # Parse tool payloads
                list_tables_payload, query_db_payload, query_db_sql = _collect_tool_payloads(result)

                # 1) Show list_tables (if present)
                if list_tables_payload:
                    display_tables_list(list_tables_payload)

                # 2) If agent already executed query_db, show it
                if query_db_payload:
                    if query_db_sql:
                        print(f"\n🔍 SQL executed (by agent): {query_db_sql}")
                    display_table_from_json(query_db_payload)
                else:
                    # 3) Agent stopped after list_tables → auto-follow-up here
                    if list_tables_payload and list_tables_payload.get("tables"):
                        tables = list_tables_payload["tables"]
                        best = _pick_closest_table(user_input, tables)
                        if best:
                            sql = f"SELECT * FROM {best} LIMIT 5"
                            print(f"\n🤖 No follow-up from agent. Auto-querying closest match: {best}")
                            print(f"🔍 SQL executed (auto): {sql}")
                            # call server directly
                            auto = await client.call_tool("query_db", {"sql": sql})
                            # Normalize response for printer
                            payload = None
                            if hasattr(auto, 'data') and auto.data:
                                payload = auto.data
                            elif hasattr(auto, "structuredContent") and auto.structuredContent:
                                payload = auto.structuredContent
                            elif hasattr(auto, "content") and auto.content:
                                payload = auto.content[0].text if isinstance(auto.content, list) else str(auto.content)
                            # If payload is stringified JSON, parse it
                            if isinstance(payload, str):
                                try:
                                    payload = json.loads(payload)
                                except Exception:
                                    payload = None
                            if isinstance(payload, dict) and payload.get("success"):
                                display_table_from_json(payload)
                            else:
                                print("ℹ️ Auto query did not return a valid payload.")
                        else:
                            print("\nℹ️ Couldn't infer a close table name from your request.")

                # Print final AI message (if any)
                messages = getattr(result, "messages", [])
                final_response = None
                for msg in reversed(messages):
                    msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
                    if 'ai' in msg_type and hasattr(msg, 'content') and msg.content:
                        final_response = msg.content
                        break
                
                if final_response:
                    print("\n🤖 Agent Response:")
                    print(final_response)

            except KeyboardInterrupt:
                print("\n👋 Goodbye!")
                break
            except Exception as e:
                print(f"\n❌ Unexpected error: {e}")
                import traceback
                print("Debug traceback:")
                traceback.print_exc()

# -----------------------------
# Run main
# -----------------------------
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(main())
        else:
            raise


🔧 Available MCP Tools:
- list_tables: List all table names in the connected PostgreSQL database.
- query_db: Execute a SELECT SQL query against PostgreSQL.

Returns:
  - preview_rows / preview_columns (server-limited to PREVIEW_* constants)
  - total_rows (true count ignoring LIMIT)
  - total_columns
  - _debug_info

🤖 Agent ready! Ask SQL questions.
Type 'exit', 'quit', or 'q' to stop.


/usr/local/lib/python3.10/site-packages/langchain_core/tools/base.py:1319: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  fields = getattr(cls, "model_fields", {}) or getattr(cls, "__fields__", {})



Enter your SQL query:  show me the content of department


🤔 Thinking...

📚 Available tables:
   1. public.departments
   2. public.employees
   3. public.projects

Total: 3 table(s)

📊 Query Result (preview):
+-----+---------------+-----------------+------------+------------+-----------+-----+
|     | department_id | department_name | manager_id |  location  |  budget   | ... |
+-----+---------------+-----------------+------------+------------+-----------+-----+
|  1  |       1       |  Department 1   |     38     | Location 7 | 365812.04 | ... |
|  2  |       2       |  Department 2   |     48     | Location 4 | 202327.53 | ... |
|  3  |       3       |  Department 3   |     43     | Location 4 | 853790.84 | ... |
|  4  |       4       |  Department 4   |     21     | Location 4 | 19482.89  | ... |
|  5  |       5       |  Department 5   |     49     | Location 8 | 619989.13 | ... |
| ... |      ...      |       ...       |    ...     |    ...     |    ...    | ... |
+-----+---------------+-----------------+------------+------------+--------


Enter your SQL query:  show me the content of employee


🤔 Thinking...

📚 Available tables:
   1. public.departments
   2. public.employees
   3. public.projects

Total: 3 table(s)

📊 Query Result (preview):
+-----+--------+------------+-----------+----------------------------+------------+-----+
|     | emp_id | first_name | last_name |           email            | hire_date  | ... |
+-----+--------+------------+-----------+----------------------------+------------+-----+
|  1  |   1    |   Alice    |  Johnson  | alice.johnson@example.com  | 2018-03-12 | ... |
|  2  |   2    |    Bob     |   Smith   |   bob.smith@example.com    | 2019-07-01 | ... |
|  3  |   3    |  Charlie   |   Davis   | charlie.davis@example.com  | 2020-01-20 | ... |
|  4  |   4    |   Diana    | Martinez  | diana.martinez@example.com | 2017-11-05 | ... |
|  5  |   5    |    Evan    |   Brown   |   evan.brown@example.com   | 2021-06-15 | ... |
| ... |  ...   |    ...     |    ...    |            ...             |    ...     | ... |
+-----+--------+------------+----------


Enter your SQL query:  exit


👋 Goodbye!
